In [4]:
import pandas as pd

In [6]:
df = pd.read_csv("gs://agntworks-data-dev/sandbox/experiments/airports.csv")

In [7]:
from math import radians, sin, cos, sqrt, atan2

# load airport data

# keep valid ICAOs
df = df[df["ident"].str.len() == 4].copy()
df = df.rename(columns={"ident": "icao"})

# haversine distance
def dist(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

# market cluster anchors (balanced ~50 clusters possible; sample core set)
clusters = {
    "NEW_YORK_CLUSTER": (40.7, -74.0),
    "BOSTON_CLUSTER": (42.36, -71.0),
    "WASHINGTON_DC_CLUSTER": (38.9, -77.0),
    "MIAMI_CLUSTER": (25.8, -80.3),
    "ORLANDO_CLUSTER": (28.5, -81.3),
    "ATLANTA_CLUSTER": (33.6, -84.4),
    "CHICAGO_CLUSTER": (41.9, -87.6),
    "DALLAS_CLUSTER": (32.9, -97.0),
    "HOUSTON_CLUSTER": (29.7, -95.3),
    "DENVER_CLUSTER": (39.8, -104.7),
    "LOS_ANGELES_CLUSTER": (34.05, -118.25),
    "SAN_FRANCISCO_CLUSTER": (37.6, -122.3),
    "SAN_DIEGO_CLUSTER": (32.7, -117.2),
    "SEATTLE_CLUSTER": (47.45, -122.3),
    "PHOENIX_CLUSTER": (33.4, -112.0),
    "LAS_VEGAS_CLUSTER": (36.1, -115.2),
    "ASPEN_CLUSTER": (39.2, -106.8),
    "VAIL_CLUSTER": (39.6, -106.4),
    "JACKSON_HOLE_CLUSTER": (43.6, -110.7),
    "HONOLULU_CLUSTER": (21.3, -157.8),
    "NASSAU_CLUSTER": (25.0, -77.4),
    "CANCUN_CLUSTER": (21.0, -86.9),
    "CABO_CLUSTER": (23.1, -109.7),
    "PUNTA_CANA_CLUSTER": (18.6, -68.4),
    "SAN_JUAN_CLUSTER": (18.4, -66.0),
    "LONDON_CLUSTER": (51.5, -0.1),
    "PARIS_CLUSTER": (49.0, 2.5),
    "MADRID_CLUSTER": (40.5, -3.6),
    "BARCELONA_CLUSTER": (41.3, 2.1),
    "ROME_CLUSTER": (41.8, 12.3),
    "MILAN_CLUSTER": (45.6, 8.7),
    "DUBAI_CLUSTER": (25.2, 55.3),
    "TOKYO_CLUSTER": (35.5, 139.8),
    "SINGAPORE_CLUSTER": (1.3, 103.8),
    "DELHI_CLUSTER": (28.6, 77.2),
}

def assign_cluster(row):
    lat, lon = row["latitude_deg"], row["longitude_deg"]

    if pd.isna(lat) or pd.isna(lon):
        return "OTHER_CLUSTER"

    best = None
    best_d = 1e9

    for name, (clat, clon) in clusters.items():
        d = dist(lat, lon, clat, clon)
        if d < best_d:
            best_d = d
            best = name

    # radius threshold (tuneable)
    return best if best_d < 500 else "OTHER_CLUSTER"

df["cluster"] = df.apply(assign_cluster, axis=1)

# final output
out = df[["icao", "cluster"]].drop_duplicates().sort_values("icao")
out.to_csv("icao_cluster.csv", index=False)

print("Done: icao_cluster.csv created")

Done: icao_cluster.csv created


In [8]:
out.to_csv("gs://agntworks-data-dev/sandbox/experiments/icao_cluster.csv", index=False)

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)
